# Concert Med Ops — Arize Phoenix Clinical Evaluations (Hackathon Track)

This notebook evaluates the base Gemma 4 model against our fine-tuned `concert-med-tox` model.
It uses **Arize Phoenix** for observability and **Gemini 1.5 Pro** as an LLM-as-a-judge to track:
1. Clinical Correctness
2. Hallucinations

**Prerequisites:**
- Set `PHOENIX_API_KEY` in Kaggle Secrets (or provide it below).
- Set `GEMINI_API_KEY` in Kaggle Secrets.
- Set `HF_TOKEN` in Kaggle Secrets (if needed for pulling the model).


In [ ]:
# Install dependencies
!pip install -q unsloth datasets transformers accelerate bitsandbytes
!pip install -q arize-phoenix[evals] openinference-instrumentation openinference-instrumentation-google-genai google-genai
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ["PHOENIX_API_KEY"] = user_secrets.get_secret("PHOENIX_API_KEY")
    os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("Secrets loaded from Kaggle!")
except Exception as e:
    print("Kaggle secrets not available. Please set environment variables manually.")


## 1. Setup Arize Phoenix Tracing

In [ ]:
import phoenix as px
from phoenix.otel import register

# Configure Phoenix to send traces to Phoenix Cloud
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={os.environ.get('PHOENIX_API_KEY')}"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"

# Initialize tracing
tracer_provider = register(
  project_name="concert-med-evals",
  endpoint="https://app.phoenix.arize.com/v1/traces"
)

# Launch Phoenix session (this will output a link to your Phoenix Cloud dashboard)
session = px.Client()
print(f"Phoenix Dashboard URL: {session.url}")


## 2. Load Evaluation Data
We'll pick a few hard clinical scenarios from our `training_data.jsonl` (or define them here for the hackathon eval).

In [ ]:
EVAL_PROMPTS = [
    {
        "input": "A 22-year-old on sertraline presents with temperature 104°F, clonus, and agitation after taking MDMA. Is this serotonin syndrome or heat stroke? What do I do first?",
        "expected": "Recognize serotonin syndrome (clonus is key). Treat with benzodiazepines and rapid cooling. Avoid haloperidol."
    },
    {
        "input": "Patient is unresponsive with pinpoint pupils and RR of 5. Friends say he snorted something. Walk me through naloxone dosing.",
        "expected": "Identify opioid overdose. Airway management and ventilations first. Intranasal or IV naloxone, titrate to effect."
    },
    {
        "input": "An MDMA user drank lots of water but is now confused and has a headache. Should I give IV fluids?",
        "expected": "Recognize severe hyponatremia. Fluid restriction is critical. Do NOT give hypotonic IV fluids. Give hypertonic saline (3%) for neurological symptoms."
    }
]

SYSTEM_PROMPT = (
    "You are Concert Med Ops AI, a clinical decision support assistant for supervising physicians, "
    "paramedics, and harm reduction volunteers at live music events and EDM festivals. "
    "Provide rapid, actionable guidance. For life-threatening emergencies, activate EMS immediately. "
    "\n\n⚠️ DISCLAIMER: AI-generated guidance. Not a substitute for licensed medical care."
)


## 3. Define Gemini Evaluator (LLM-as-a-Judge)
We use Gemini 1.5 Pro to evaluate responses based on correctness and hallucination.

In [ ]:
from google import genai
from pydantic import BaseModel, Field

gemini_client = genai.Client()

class EvalResult(BaseModel):
    is_correct: int = Field(description="1 if the answer is clinically correct and safe, 0 otherwise.")
    hallucination_score: int = Field(description="1 if the answer hallucinated fake medical facts or non-existent protocols, 0 if it is grounded.")
    reasoning: str = Field(description="Reasoning for the scores.")

def evaluate_response(user_input: str, generated_response: str, expected_guidance: str) -> EvalResult:
    prompt = f"""You are an expert medical toxicology evaluator.
Evaluate the following AI-generated response for a medical emergency at a music festival.

User Query: {user_input}
Expected Clinical Guidance: {expected_guidance}

AI Generated Response:
{generated_response}

Evaluate:
1. is_correct: Does the AI provide the expected clinical guidance safely without dangerous omissions? (1=Yes, 0=No)
2. hallucination_score: Did the AI make up fake medical facts or refer to non-existent protocols? (1=Yes, 0=No)
"""
    
    response = gemini_client.models.generate_content(
        model='gemini-1.5-pro',
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=EvalResult,
            temperature=0.0
        ),
    )
    return response.parsed


## 4. Run Base Model (Gemma 4)
Load the base model and generate answers.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

# Load Base Model
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def generate_answer(model, user_input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(input_ids=input_ids, max_new_tokens=512, temperature=0.7, do_sample=True, top_p=0.9, use_cache=True)
    generated = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


In [ ]:
# Instrument evaluation function manually or let Gemini auto-instrument if set up globally.
# For simplicity, we just log results to Phoenix DataFrame.

base_results = []
for prompt_data in EVAL_PROMPTS:
    print(f"Testing Base Model on: {prompt_data['input'][:50]}...")
    
    # Generate
    response = generate_answer(base_model, prompt_data['input'])
    
    # Evaluate
    eval_res = evaluate_response(prompt_data['input'], response, prompt_data['expected'])
    
    base_results.append({
        "model": "base_gemma4",
        "input": prompt_data['input'],
        "response": response,
        "is_correct": eval_res.is_correct,
        "hallucination": eval_res.hallucination_score,
        "reasoning": eval_res.reasoning
    })
    
    # Log to Phoenix Client explicitly if desired
    # session.log_evaluations(...) # simplified for notebook display

import pandas as pd
df_base = pd.DataFrame(base_results)
df_base


## 5. Run Fine-Tuned Model (`concert-med-tox`)
Now apply our LoRA adapters trained on the Concert Med Ops KB.

In [ ]:
# To run the fine-tuned version, you would load the LoRA weights. 
# Assuming you exported them to a local directory or HuggingFace:
# LORA_DIR = "/kaggle/working/concert-med-tox-checkpoints/checkpoint-XXX"
LORA_DIR = "YOUR_HF_USERNAME/concert-med-tox-gemma3-4b" # Replace with actual path or HF repo

try:
    tuned_model, tuned_tokenizer = FastLanguageModel.from_pretrained(
        model_name=LORA_DIR, # Loads base + LoRA
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(tuned_model)
    tuned_tokenizer = get_chat_template(tuned_tokenizer, chat_template="gemma-4")
    
    tuned_results = []
    for prompt_data in EVAL_PROMPTS:
        print(f"Testing Tuned Model on: {prompt_data['input'][:50]}...")
        response = generate_answer(tuned_model, prompt_data['input'])
        eval_res = evaluate_response(prompt_data['input'], response, prompt_data['expected'])
        
        tuned_results.append({
            "model": "concert-med-tox",
            "input": prompt_data['input'],
            "response": response,
            "is_correct": eval_res.is_correct,
            "hallucination": eval_res.hallucination_score,
            "reasoning": eval_res.reasoning
        })
        
    df_tuned = pd.DataFrame(tuned_results)
    display(df_tuned)
except Exception as e:
    print(f"Could not load LoRA model. Did you finish training? Error: {e}")


## 6. Analyze Head-to-Head
Check the Phoenix Dashboard for trace data, and compare the DataFrames above.